In [ ]:
# ----------------------------
# CNN PINN simulator
# ----------------------------

# Import necessary libraries
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.ndimage import distance_transform_edt

In [ ]:
# ----------------------------
# CNN PINN Block Definitions
# ----------------------------

class ConvBlock(nn.Module):
        """Two 3x3 convs, each followed by GroupNorm + SiLU."""
        def __init__(self, in_ch, out_ch, groups=8):
            super().__init__()
            groups = min(groups, out_ch)
            self.net = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.GroupNorm(groups, out_ch),
                nn.SiLU(),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.GroupNorm(groups, out_ch),
                nn.SiLU(),
            )
        
        def forward(self, x):
            return self.net(x)
    
class DownBlock(nn.Module):
    """Stride-2 conv downsample + ConvBlock.
    Learned (strided-conv) downsampling instead of max-pooling -- it preserves more boundary-layer detail,
    which matters for near-wall gradients that impact physics loss"""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv2d(in_ch, in_ch, 4, stride=2, padding=1)
        self.block = ConvBlock(in_ch, out_ch)
    
    def forward(self, x):
        return self.block(self.down(x))

class UpBlock(nn.Module):
    """Transpose-conv upsample, concat skip connection, ConvBlock."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)
        self.block = ConvBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            # guards against off-by-one size mismatches on odd input dims
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)
    
# ----------------------------
# Spectral Convolution Layer for the bottleneck to solve pressure across the whole domain
# ----------------------------

class SpectralConv2d(nn.Module):
    """2D Fourier layer. This is the spectral convolution layer used in the bottleneck of the CNN PINN architecture."""
    def __init__(self, in_ch, out_ch, modes_h, modes_w):
        super().__init__()
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.modes_h = modes_h
        self.modes_w = modes_w
        self.scale = (1.0 / (in_ch * out_ch))
        self.weights = nn.Parameter(self.scale * torch.rand(in_ch, out_ch, modes_h, modes_w, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft_rfft2(x, norm="ortho") # (B, C, H, W//2+1), complex

        out_ft = torch.zeros(B, self.out_ch, H, W // 2 + 1, dtype=torch.cfloat, device=x.device)
        mh = min(self.modes_h, H)
        mw = min(self.modes_w, W // 2 + 1)

        out_ft[:, :, :mh, :mw] = torch.einsum(
            "bixy,ioxy->boxy", x_ft[:, :, :mh, :mw], self.weights[:, :, :mh, :mw]
        )

        return torch.fft.irfft2(out_ft, s=(H, W), norm="ortho")
    
class FNOBlock(nn.Module):
    """Fourier Neural Operator block. This is the bottleneck of the CNN PINN architecture."""
    def __init__(self, channels, modes_h=16, modes_w=16):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_h, modes_w)
        self.pointwise = nn.Conv2d(channels, channels, 1)
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.act = nn.SiLU()

    def forward(self, x):
        out = self.spectral(x) + self.pointwise(x)
        return self.act(self.norm(out))
        

In [ ]:
class CNN_PINN(nn.Module):
    """
    U-Net encoder-deecoder with an FNO bottleneck.
    
    Input: (B, 4, H, W) -- occupancy, SDF, x-coord, y-coord
    Output: (B, 4, H, W) -- u, v, p, T (fluid region values only, masked in solid region)

    Sized for H=100, W = 200 (25x50 bottleneck, small enough for FFT-based spectral convs to be cheap)
    """
    def __init__(self, in_ch=4, base_ch=32, n_fno_blocks=4, fno_modes=16):
        super().__init__()

        self.stem = ConvBlock(in_ch, base_ch)               # H x   W
        self.down1 = DownBlock(base_ch, base_ch * 2)        # H/2 x W/2
        self.down2 = DownBlock(base_ch * 2, base_ch * 4)    # H/4 x W/4

        self.bottleneck = nn.Sequential(
            *[FNOBlock(base_ch * 4, fno_modes, fno_modes) for _ in range(n_fno_blocks)]
        )

        self.up2 = UpBlock(base_ch * 4, base_ch * 2, base_ch * 2)  # H/2 x W/2
        self.up1 = UpBlock(base_ch * 2, base_ch, base_ch)
        self.head = nn.Conv2d(base_ch, 4, kernel_size=1)  # u, v, p, T

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.bottleneck(s2)
        u2 = self.up2(b, s1)
        u1 = self.up1(u2, s0)
        out = self.head(u1)
        return out

In [ ]:
class SpatialDerivative(nn.Module):
    """
    Calculates spatial derivative operators via fixed conv2d kernels using central difference approximations.
    Replicate-padded to get a valid derivative at the boundary. This is used to compute the physics loss for the PINN.
    """
    def __init__(self, dx: float, dy: float):
        super().__init__()
        assert abs(dx - dy) < 1e-9, "assumes square pixels (dx == dy)"
        self.h = dx
        
        ddx = torch.tensor([[0, 0, 0], [-1, 0, 1], [0, 0, 0]], dtype=torch.float32) / (2 * dx)
        ddy = torch.tensor([[0, -1, 0], [0, 0, 0], [0, 1, 0]], dtype=torch.float32) / (2 * dy)
        lap = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32) / (dx * dy)

        self.register_buffer("ddx", ddx.view(1, 1, 3, 3))
        self.register_buffer("ddy", ddy.view(1, 1, 3, 3))
        self.register_buffer("lap", lap.view(1, 1, 3, 3))

    def _conv_deriv(self, f, kernel):
        f = F.pad(f, (1, 1, 1, 1), mode="replicate")
        return F.conv2d(f, kernel)

    def d_dx(self, f):
        return self._conv_deriv(f, self.ddx)
    
    def d_dy(self, f):
        return self._conv_deriv(f, self.ddy)
    
    def laplacian(self, f):
        return self._conv_deriv(f, self.lap)

In [ ]:
class PhysicsLoss(nn.Module):
    """
    Computes PDE residual + boundary-condition losses for a single fixed
    set of fluid properties / boundary conditions.

    forward() returns a dict of individual loss terms, which can be weighted and summed to form the total loss.
    """

    def __init__(self, fluid_properties: dict, boundary_conditions: dict, domain_size=(2.0, 1.0), grid_shape=(200, 100)):
        super().__init__()
        W, H = grid_shape
        dx = domain_size[0] / W
        dy = domain_size[1] / H
        self.deriv = SpatialDerivative(dx, dy)

In [ ]:
class PINN_Simulator(nn.Module):
    super().__init__()

    def build_input_tensor(occupancy_grid: np.ndarray, domain_size=(2.0,1.0)) -> torch.Tensor:
        """
        Convert a binary occupancy grid (1 = solid, 0 = fluid) into the
        multi-channel input tensor for the network.
    
        Channels:
            0: occupancy            (0 = fluid, 1 = solid)
            1: signed distance fn   (positive in fluid, negative in solid),
                                    normalized by the domain diagonal
            2: normalized x coord   in [0, 1]
            3: normalized y coord   in [0, 1]
    
        Coordinate + SDF channels matter because plain convolutions are
        translation-equivariant, but inlet/outlet/wall locations are fixed in
        the domain -- the network needs to know *where* it is, and the SDF gives
        a smooth signal near walls instead of a hard 0/1 jump.
    
        Args:
            occupancy_grid: (H, W) array, 1 = solid, 0 = fluid
            domain_size: (Lx, Ly) physical domain size in meters
    
        Returns:
            Tensor of shape (4, H, W), float32
        """
        occ = occupancy_grid.astype(np.float32)
        H, W = occ.shape
    
        dx = domain_size[0] / W
        dy = domain_size[1] / H
        assert abs(dx - dy) < 1e-9, "build_input_tensor assumes square pixels (dx == dy)"
    
        dist_to_solid = distance_transform_edt(occ == 0)  # fluid px -> nearest solid px
        dist_to_fluid = distance_transform_edt(occ == 1)  # solid px -> nearest fluid px
        sdf_px = np.where(occ == 0, dist_to_solid, -dist_to_fluid)
        sdf_m = sdf_px * dx
        diag = np.sqrt(domain_size[0] ** 2 + domain_size[1] ** 2)
        sdf_norm = (sdf_m / diag).astype(np.float32)
    
        y_coords, x_coords = np.meshgrid(
            np.linspace(0.0, 1.0, H, dtype=np.float32),
            np.linspace(0.0, 1.0, W, dtype=np.float32),
            indexing="ij",
        )
    
        input_np = np.stack([occ, sdf_norm, x_coords, y_coords], axis=0)
        return torch.from_numpy(input_np)
    